In [ ]:
import random
import torch
import os
import re

import pandas as pd
import polars as pl
import numpy as np

import sys
sys.path.append('../')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

from collections import defaultdict

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric == corpus_metrics.zero_biber_distance:
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance, corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances


In [ ]:
# subject the real data to the same processing as the generated data
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()

real_dataset = pd.read_csv(f'../realNotes/makeOneBigFile/dataRealAll.csv', index_col=None)

real_dataset = real_dataset.dropna(subset='Note')
real_dataset['Note'] = [clean_note(text) for text in real_dataset['Note'].tolist()]


In [ ]:
input_generated_data_dirs = ['dataGeneration/syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes', 'dataGeneration/syntheticNotesOnline/gpt3Notes', 'dataGeneration/syntheticNotesOnline/gpt4Notes']
all_dfs = {}
for directory in input_generated_data_dirs:
    temp_dfs = []
    for file in os.listdir(f'./{directory}'):
        if not os.path.isdir(f'./{directory}/{file}'):
            temp_dfs.append(pd.read_csv(f'./{directory}/{file}'))
    all_dfs[directory.split('/')[-1]] = pd.concat(temp_dfs)

In [ ]:
dataset_metrics_averaged = {}

for dataset, temp_df in all_dfs.items():
    temp_results = defaultdict(list)
    
    for i in range(3):
        real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()[:100]
        model_sample = temp_df.sample(frac=1, random_state = i)['report'].dropna().tolist()[:100]
        temp_metrics = get_distances_from_compare_corpora(real_sample, model_sample)

        for metric, value in temp_metrics.items():
            temp_results[metric].append(value)

    dataset_metrics_averaged[dataset] = {}                                       
    for metric, values in temp_results.items():
        dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

In [ ]:
dataset = 'realToReal'
temp_results = defaultdict(list)

for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for metric, value in temp_metrics.items():
        temp_results[metric].append(value)

dataset_metrics_averaged[dataset] = {}                                       
for metric, values in temp_results.items():
    dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

dataset = 'realToReal2'
temp_results = defaultdict(list)

for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i+200)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for metric, value in temp_metrics.items():
        temp_results[metric].append(value)

dataset_metrics_averaged[dataset] = {}                                       
for metric, values in temp_results.items():
    dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

dataset = 'realToReal3'
temp_results = defaultdict(list)

for i in range(3):
    real_sample = real_dataset.sample(frac=1, random_state = i+300)['Note'].tolist()
    assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
    real_start = real_sample[:100]
    real_end = real_sample[-100:]
    temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

    for metric, value in temp_metrics.items():
        temp_results[metric].append(value)

dataset_metrics_averaged[dataset] = {}                                       
for metric, values in temp_results.items():
    dataset_metrics_averaged[dataset][metric] = sum(values) / len(values)

In [ ]:

df = pd.DataFrame.from_dict(dataset_metrics_averaged, orient='index')
df.reset_index(inplace=True)
df.rename(columns={'index': 'dataset'}, inplace=True)

df.to_csv("./realFakemetrics.csv", index=False)

In [ ]:
df